# PHYS 512 — 🎄 vs 🧠: The MNIST Challenge
**Thursday, October 29, 2026**  
**Credit:** [Maxwell A. Fine](https://afinemax.github.io/afinemax1/)

_Take Chances, make mistakes, get messy - Dr. Frizzle_

#### The challenge
Train a machine-learning model to recognize handwritten digits (0–9) from the **MNIST** dataset. Each image is a 28 × 28 grayscale image. MNIST contains 60,000 training images and 10,000 test images. This is conceptually the same as the "Galaxy Zoo" classification problem from the first machine learning lecture. 

**Best group wins Timbits!** Work in groups, experiment with model choice, loss function, augmentation, and hyperparameters, and give a short presentation at the end explaining your approach and results.

**Your choices:** Try a convolutional neural network (CNN), a random forest (RF), or an ensemble of both. How will you train? Which hyperparameters matter? What errors does your model make?

**Rules:** Tune using the validation set. Keep the test set untouched until final evaluation. Compare models on the same split and random seed. Record your best settings and accuracy.

**At the end:** Each group presents its model, training choices, validation performance, and one thing they learned. 

**No Cheating:** If your model is **_100% accurate_** on the test data set, you have data leakage!


## 1. Install dependencies
Run once if your environment does not already have these packages. Restart the kernel if needed. ( It does no harm to re-run it)

In [ ]:
pip install -q torch torchvision scikit-learn matplotlib numpy

## 2. Imports, reproducibility, and MNIST download
- MNIST download is so common, its apart of common machine learning python packages.
- Don't change the seed, this ensures everyone has the same images in the train, validate, and test categories

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

SEED = 512
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

raw_train = datasets.MNIST(root="./data", train=True, download=True)
raw_test = datasets.MNIST(root="./data", train=False, download=True)
indices = rng.permutation(len(raw_train))
train_idx, val_idx = indices[:50000], indices[50000:]
print(f"Train: {len(train_idx):,} | Validation: {len(val_idx):,} | Test: {len(raw_test):,}")

## 3. Explore the data
What makes some digits harder to distinguish than others?

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(11, 6))
for ax, i in zip(axes.flat, rng.choice(len(raw_train), 18, replace=False)):
    image, label = raw_train[int(i)]
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Fill in:
- Make a bar plot of the number of digits in each class
- Any dramatically over or under represented?

In [ ]:
#TODO: your code here

## 4. Prepare datasets and loaders
The CNN receives images shaped `(batch, 1, 28, 28)`; the random forest receives flattened vectors of 784 pixel intensities. 

**Experiment:** Enable a small random rotation or translation. Too much augmentation can change a digit's meaning!

In [ ]:
# TODO: Try changing augmentation parameters (or removing augmentation).
# TODO: what do the degrees, translate do? What does the 0.1307 do?

train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=8, translate=(0.08, 0.08)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

cnn_train = Subset(datasets.MNIST("./data", train=True, transform=train_transform), train_idx.tolist())
cnn_val = Subset(datasets.MNIST("./data", train=True, transform=eval_transform), val_idx.tolist())
cnn_test = datasets.MNIST("./data", train=False, transform=eval_transform)

BATCH_SIZE = 128  # TODO: Tune me
train_loader = DataLoader(cnn_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(cnn_val, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(cnn_test, batch_size=256, shuffle=False, num_workers=0)

# RF uses original unaugmented pixels; keep the same split as the CNN.
X = raw_train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y = raw_train.targets.numpy()
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test = raw_test.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = raw_test.targets.numpy()

## 5A. CNN approach — build your network
A CNN learns spatial features through convolutional filters. Start with the baseline below, then change the number of filters, layers, dropout, or activation functions.

**TODO:** What happens when you add another convolutional block?
**TODO:** Is more neurons or more fully connected layers better?
**TODO:** Can you make a diagram of what the model looks like?
**TODO:** Change the _hyper-paramters_ of the model around

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64), #TODO: can you figure out the same of the fully connected layers?
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

cnn = SimpleCNN().to(DEVICE)
print(cnn)

## 5B. Alternative: Random Forest
A random forest combines many decision trees. Unlike the CNN, it does not explicitly preserve the 2D geometry of the image.

**TODO:** Experiment with `n_estimators`, `max_depth`, and `max_features`. How does training time compare?

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,  # TODO: Tune
    max_depth=20,      # TODO: Try None or other values
    n_jobs=-1,
    random_state=SEED,
)


## 6a. Train the CNN
Try changing the learning rate, number of epochs, optimizer, or batch size. Run the cell below if your group chooses the CNN approach.

In [ ]:
LEARNING_RATE = 1e-3  # TODO: Tune, this controls how much we multiply the gradient IE it effects the step sizes when the model learns
EPOCHS = 3            # TODO: Tune
optimizer = torch.optim.Adam(cnn.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

history = []
for epoch in range(EPOCHS):
    cnn.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = cnn(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)
    mean_loss = running_loss / len(cnn_train)
    history.append(mean_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS}: training loss = {mean_loss:.4f}")

plt.plot(range(1, EPOCHS + 1), history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.show()

It is computationally expensive to train, cheap to run. More then 1 member of your group should be training models. 

 ## 6a. Train the Random Forrest

In [ ]:
# TODO: Try a smaller subset for faster experiments.
RF_TRAIN_SIZE = len(X_train)

rf.fit(X_train[:RF_TRAIN_SIZE], y_train[:RF_TRAIN_SIZE])

print(f"Trained {rf.n_estimators} trees on {RF_TRAIN_SIZE:,} images.")

# Quick validation check
rf_train_val_pred = rf.predict(X_val)
print(f"Random Forest validation accuracy: {accuracy_score(y_val, rf_train_val_pred):.2%}")


# TODO: Make a "learning curve" plot for the random forrest, accuracy vs the number of trees
# Note: Randomforests don't learn in the same way as a neural-net


## 7. Validation — compare your approaches
Use this section to evaluate whichever models you trained. Do **not** use the test set for hyperparameter tuning.

In [ ]:
@torch.no_grad()  # Disable gradient calculations during evaluation
def cnn_probabilities(model, loader):
    model.eval()
    all_probs = []

    for images, _ in loader:
        logits = model(images.to(DEVICE))
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

    return np.concatenate(all_probs)


# CNN validation
cnn_val_probs = cnn_probabilities(cnn, val_loader)
cnn_val_pred = cnn_val_probs.argmax(axis=1)
print(f"CNN validation accuracy: {accuracy_score(y_val, cnn_val_pred):.2%}")


# Random Forest validation

rf_val_probs = rf.predict_proba(X_val)
rf_val_pred = rf_val_probs.argmax(axis=1)
print(f"Random Forest validation accuracy: {accuracy_score(y_val, rf_val_pred):.2%}")


## 8a. Ensemble — can two models beat one?
A simple **soft-voting ensemble** averages predicted class probabilities:

$$P(y=k\mid x)=w P_{\rm CNN}(y=k\mid x)+(1-w)P_{\rm RF}(y=k\mid x).$$

**TODO:** Train both models, set, rerun validation, and experiment with `w` from 0 to 1. Choose the weight using validation accuracy only.

In [ ]:
ENSEMBLE_WEIGHT = 0.5  # TODO: Tune on validation data
if cnn_val_probs is not None and rf_val_probs is not None:
    ensemble_val_probs = (ENSEMBLE_WEIGHT * cnn_val_probs
                          + (1 - ENSEMBLE_WEIGHT) * rf_val_probs)
    ensemble_val_pred = ensemble_val_probs.argmax(axis=1)
    print("Ensemble validation accuracy:", accuracy_score(y_val, ensemble_val_pred))
else:
    print("Train and validate both models to try an ensemble.")

## 8b Efficiency
It is common to want to know how many _tunable parameters_ a model has, these are different from the hyper-parameters. A more complex model costs more to train, and takes longer to run then a simple model. The more complex model might be more accurate, but not always, if you use a clever data augmentation scheme, an elegant simple model will often beat a longer trained, or more complex model. 

In [ ]:
# CNN: Count trainable parameters
cnn_params = sum(
    p.numel() for p in cnn.parameters() if p.requires_grad
)

# Random Forest: Count total tree nodes
rf_nodes = sum(
    tree.tree_.node_count for tree in rf.estimators_
)

print(f"CNN trainable parameters: {cnn_params:,}")
print(f"Random Forest nodes:      {rf_nodes:,}")

## 9. Inspect mistakes
Which digits does your best model confuse? Can you explain why? **TODO**: should these digits be penalized more in training? Should more of these digits appear in the training? 

In [ ]:
if cnn_val_probs is not None or rf_val_probs is not None:
    predictions = (ensemble_val_pred if cnn_val_probs is not None and rf_val_probs is not None
                   else cnn_val_probs.argmax(axis=1) if cnn_val_probs is not None
                   else rf_val_probs.argmax(axis=1))
    ConfusionMatrixDisplay.from_predictions(y_val, predictions, cmap="Blues", values_format="d")
    plt.title("Validation confusion matrix")
    plt.show()
    mistakes = np.flatnonzero(predictions != y_val)
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for ax, j in zip(axes.flat, mistakes[:10]):
        ax.imshow(X_val[j].reshape(28, 28), cmap="gray")
        ax.set_title(f"True {y_val[j]}, predicted {predictions[j]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

**TODO** Can you explain how a random forrest works? 
**TODO** Can you give a conceptual overview of how a CNN works?
**TODO** Experiment with changing the hyper-parameters, training routine, of the random forrest, and CNN. 
**TODO** From your experiments come up with your best (ensemble) model 


**TODO** You are not limited by a CNN, or random forest, if you want to try another model you can (and you earn a timbit!)



## 10. Final test evaluation — run once at the end
After choosing your model and hyperparameters on the validation set, evaluate on the held-out test set. For a strictly fair final competition, the instructor should control when this cell is run (call me over!). The test set must not guide further tuning.

In [ ]:
# # ============================================================
# # FINAL TEST — Run only after all hyperparameter tuning!
# # TODO: Call the instructor over when you are ready to run this
# # TODO: You will need to update this cell with your groups models if you use different names or more models
# # ============================================================

# # Get predicted probabilities from both models
# cnn_test_probs = cnn_probabilities(cnn, test_loader)
# rf_test_probs = rf.predict_proba(X_test)

# # Combine predictions using a weighted average
# # ENSEMBLE_WEIGHT = 1.0 -> CNN only
# # ENSEMBLE_WEIGHT = 0.0 -> Random Forest only
# # ENSEMBLE_WEIGHT = 0.5 -> Equal weighting
# test_probs = (
#     ENSEMBLE_WEIGHT * cnn_test_probs
#     + (1 - ENSEMBLE_WEIGHT) * rf_test_probs
# )

# # Calculate accuracies
# cnn_accuracy = accuracy_score(y_test, cnn_test_probs.argmax(axis=1))
# rf_accuracy = accuracy_score(y_test, rf_test_probs.argmax(axis=1))
# ensemble_accuracy = accuracy_score(y_test, test_probs.argmax(axis=1))

# print(f"CNN test accuracy:           {cnn_accuracy:.2%}")
# print(f"Random Forest test accuracy: {rf_accuracy:.2%}")
# print(f"Ensemble test accuracy:      {ensemble_accuracy:.2%}")

In [ ]:
# Uncomment when your group has finished all tuning.
# cnn_test_probs = cnn_probabilities(cnn, test_loader) if USE_CNN else None
# rf_test_probs = rf.predict_proba(X_test) if USE_RF else None
# if cnn_test_probs is not None and rf_test_probs is not None:
#     test_probs = ENSEMBLE_WEIGHT * cnn_test_probs + (1 - ENSEMBLE_WEIGHT) * rf_test_probs
# elif cnn_test_probs is not None:
#     test_probs = cnn_test_probs
# else:
#     test_probs = rf_test_probs
# print("FINAL TEST ACCURACY:", accuracy_score(y_test, test_probs.argmax(axis=1)))

## 11. Mini-presentations 
Each group: **2–3 minutes**.

1. Which model(s) did you choose, and why?
2. Which hyperparameters or augmentation strategies did you try?
3. How long was your final training run?
4. What was your best **validation accuracy** (and final test accuracy)?
5. What did you learn from the errors? Did ensembling help?

**Timbits go to the highest final test accuracy.** In a tie, compare model complexity (more simple the better).